# 3.1 Обучение модели классификации учебных материалов

**Цель:** обучить модели ML для классификации учебных материалов по трём целевым переменным:
1. `parallel_cluster` — пригодность для параллельного изучения (4 класса)
2. `sequential_cluster` — пригодность для последовательного изучения (4 класса)
3. `complexity_cluster` — уровень сложности освоения (3 класса: Базовый / Средний / Продвинутый)

Для каждой цели сравниваются **4 алгоритма**: Logistic Regression, Random Forest, XGBoost, LightGBM.  
Метрики: Accuracy, Precision, Recall, F1-macro, ROC-AUC (OvR).  
Лучшая модель сохраняется в `../models/`.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os
from pathlib import Path
from datetime import datetime

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import lightgbm as lgb

from src.config import MODELS_DIR, LOGS_DIR, TARGET_PARALLEL, TARGET_SEQUENTIAL, TARGET_COMPLEXITY
from src.db import load_labelled_materials
from src.features import build_features
from src.model_registry import log_model_version

REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)
print('Импорты загружены. Python', sys.version)

## 1. Загрузка и анализ данных

In [ ]:
df = load_labelled_materials()
print(f'Загружено материалов: {len(df)}')
print(f'Колонки: {list(df.columns)}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
targets = [
    (TARGET_PARALLEL,   'Параллельное изучение', axes[0]),
    (TARGET_SEQUENTIAL, 'Последовательное изучение', axes[1]),
    (TARGET_COMPLEXITY, 'Сложность освоения', axes[2]),
]
for col, title, ax in targets:
    df[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Кластер')
    ax.set_ylabel('Количество материалов')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(REPORTS / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Engineering

Признаки: TF-IDF по тексту (300 токенов) + числовые признаки  
(word_count, avg_sentence_length, media_count, has_images, has_videos, has_questions, compliance_score, is_generated).

In [ ]:
X_sparse, tfidf_vec, scaler = build_features(df, max_tfidf=300)
print(f'Матрица признаков: {X_sparse.shape}')

## 3. Обучение и сравнение 4 алгоритмов

| # | Алгоритм | Обоснование |
|---|---------|-------------|
| 1 | **Logistic Regression** | Базовая линейная модель; устойчива на разреженных матрицах TF-IDF; быстрая, интерпретируемая |
| 2 | **Random Forest** | Ансамбль деревьев; устойчив к шуму; не требует масштабирования; даёт feature importance |
| 3 | **XGBoost** | Градиентный бустинг с регуляризацией; выигрывает у RF на табличных данных |
| 4 | **LightGBM** | Leaf-wise бустинг; быстрее XGBoost на разреженных TF-IDF матрицах |

Оценка: Stratified K-Fold CV (до 5 фолдов) по F1-macro.

In [ ]:
def get_classifiers():
    return {
        'LogisticRegression': LogisticRegression(
            max_iter=1000, C=1.0, solver='saga', random_state=42
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            eval_metric='mlogloss', random_state=42, verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1
        ),
    }

def evaluate_models(X, y, target_col):
    min_class = y.value_counts().min()
    n_splits  = min(5, int(min_class))

    records = []
    for name, clf in get_classifiers().items():
        if n_splits >= 2:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            f1_scores  = cross_val_score(clf, X, y, cv=cv, scoring='f1_macro',  n_jobs=-1)
            acc_scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy',  n_jobs=-1)
            f1_mean, f1_std = f1_scores.mean(), f1_scores.std()
            acc_mean        = acc_scores.mean()
        else:
            # Слишком мало примеров — простой hold-out
            try:
                X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
            except ValueError:
                X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
            clf.fit(X_tr, y_tr)
            y_pred  = clf.predict(X_te)
            f1_mean = f1_score(y_te, y_pred, average='macro', zero_division=0)
            acc_mean = accuracy_score(y_te, y_pred)
            f1_std  = 0.0

        records.append({'Модель': name, 'Accuracy': acc_mean,
                        'F1-macro': f1_mean, 'F1-std': f1_std})
        print(f'  {name:20s}: F1={f1_mean:.4f} ± {f1_std:.4f}')

    return pd.DataFrame(records).sort_values('F1-macro', ascending=False)

print('Функции оценки готовы.')

In [ ]:
all_results = {}
target_labels = {
    TARGET_PARALLEL:   'Параллельное изучение',
    TARGET_SEQUENTIAL: 'Последовательное изучение',
    TARGET_COMPLEXITY: 'Сложность освоения',
}

for target_col, label in target_labels.items():
    print(f'\n=== {label} ({target_col}) ===')
    y = pd.Series(pd.factorize(df[target_col])[0], index=df.index)
    results_df = evaluate_models(X_sparse, y, target_col)
    all_results[target_col] = {'label': label, 'results': results_df}
    print(results_df.to_string(index=False))

## 4. Визуализация сравнения алгоритмов

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (target_col, info) in zip(axes, all_results.items()):
    res    = info['results']
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(res))]
    bars   = ax.barh(res['Модель'], res['F1-macro'], color=colors, edgecolor='white')
    ax.set_xlim(0, 1)
    ax.set_title(info['label'], fontsize=12, fontweight='bold')
    ax.set_xlabel('F1-macro')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.4)
    for bar, val in zip(bars, res['F1-macro']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Сравнение алгоритмов по F1-macro', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Финальное обучение лучших моделей + тест на отложенной выборке

In [ ]:
def train_best_and_evaluate(X, y, best_name):
    clf = get_classifiers()[best_name]
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    except ValueError:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_te, y_pred, average='macro', zero_division=0)

    try:
        classes = sorted(y.unique())
        y_prob  = clf.predict_proba(X_te)
        auc = roc_auc_score(
            label_binarize(y_te, classes=classes), y_prob,
            multi_class='ovr', average='macro'
        ) if y.nunique() > 2 else roc_auc_score(y_te, y_prob[:, 1])
    except Exception:
        auc = float('nan')

    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-macro:  {f1:.4f}')
    print(f'  ROC-AUC:   {auc:.4f}' if auc == auc else '  ROC-AUC:   —')
    print()
    print(classification_report(y_te, y_pred, zero_division=0))

    # Финальное обучение на всех данных
    clf.fit(X, y)
    return clf, {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': auc}

best_models   = {}
holdout_metrics = {}

for target_col, info in all_results.items():
    best_name = info['results'].iloc[0]['Модель']
    print(f'\n>>> {info["label"]} — лучший: {best_name}')
    y = pd.Series(pd.factorize(df[target_col])[0], index=df.index)
    clf, metrics = train_best_and_evaluate(X_sparse, y, best_name)
    best_models[target_col]   = (clf, best_name)
    holdout_metrics[target_col] = metrics

## 6. Матрицы ошибок

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (target_col, info) in zip(axes, all_results.items()):
    clf, best_name = best_models[target_col]
    y = pd.Series(pd.factorize(df[target_col])[0], index=df.index)
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(X_sparse, y, test_size=0.2, stratify=y, random_state=42)
    except ValueError:
        X_tr, X_te, y_tr, y_te = train_test_split(X_sparse, y, test_size=0.2, random_state=42)
    clf_tmp = get_classifiers()[best_name]
    clf_tmp.fit(X_tr, y_tr)
    ConfusionMatrixDisplay.from_estimator(clf_tmp, X_te, y_te, ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{info["label"]}\n({best_name})', fontsize=10)

plt.suptitle('Матрицы ошибок на отложенной выборке (20%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Сохранение моделей

In [ ]:
version = datetime.now().strftime('%Y%m%d_%H%M%S')
version_dir = MODELS_DIR / f'v_{version}'
version_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf_vec, version_dir / 'tfidf_vectorizer.joblib')
joblib.dump(scaler,    version_dir / 'scaler.joblib')

model_meta = {
    'version': version,
    'trained_at': datetime.now().isoformat(),
    'n_materials': len(df),
    'models': {}
}

for target_col, (clf, algo_name) in best_models.items():
    fname = f'{target_col}_best.joblib'
    joblib.dump(clf, version_dir / fname)
    model_meta['models'][target_col] = {
        'algorithm': algo_name,
        'file': str(version_dir / fname),
        'metrics': holdout_metrics[target_col]
    }
    print(f'  Сохранено: {version_dir / fname}')

with open(version_dir / 'meta.json', 'w', encoding='utf-8') as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

(MODELS_DIR / 'current_version.txt').write_text(version, encoding='utf-8')

log_path = LOGS_DIR / 'training_log.jsonl'
with open(log_path, 'a', encoding='utf-8') as f:
    f.write(json.dumps(model_meta, ensure_ascii=False) + '\n')

try:
    log_model_version(model_meta, str(version_dir))
    print('??????????? ?????? ? ??: public.module_v_model_versions')
except Exception as exc:
    print(f'??????????????: ?? ??????? ???????????? ?????? ? ??: {exc}')

print(f'\nВерсия: {version}')
print(f'Мета:   {version_dir / "meta.json"}')
print(f'TF-IDF: {version_dir / "tfidf_vectorizer.joblib"}')
print(f'Scaler: {version_dir / "scaler.joblib"}')

## 8. Итоговая таблица метрик

In [ ]:
rows = []
for target_col, metrics in holdout_metrics.items():
    _, algo = best_models[target_col]
    rows.append({
        'Задача': target_labels[target_col],
        'Алгоритм': algo,
        'Accuracy':  f'{metrics["accuracy"]:.4f}',
        'Precision': f'{metrics["precision"]:.4f}',
        'Recall':    f'{metrics["recall"]:.4f}',
        'F1-macro':  f'{metrics["f1"]:.4f}',
        'ROC-AUC':   f'{metrics["roc_auc"]:.4f}' if metrics['roc_auc'] == metrics['roc_auc'] else '—',
    })

summary_df = pd.DataFrame(rows)
print('=== ИТОГОВЫЕ МЕТРИКИ ЛУЧШИХ МОДЕЛЕЙ ===')
print(summary_df.to_string(index=False))
summary_df.to_csv(REPORTS / 'model_metrics_summary.csv', index=False, encoding='utf-8')
print('\nТаблица сохранена: reports/model_metrics_summary.csv')

## Выводы и обоснование выбора алгоритмов

**Logistic Regression** — базовая линейная модель. Эффективна на разреженных TF-IDF матрицах, интерпретируема. Служит baseline.

**Random Forest** — ансамблевый метод. Устойчив к выбросам, не требует масштабирования числовых признаков. Даёт `feature_importances_` для интерпретации.

**XGBoost** — градиентный бустинг с L1/L2 регуляризацией. На большинстве табличных задач превосходит Random Forest по F1 и ROC-AUC.

**LightGBM** — leaf-wise бустинг. Значительно быстрее XGBoost на разреженных входных данных (TF-IDF), что критично при частом переобучении в ноутбуке 3.2.

Финальный выбор — алгоритм с наибольшим F1-macro на кросс-валидации, проверенный на отложенной выборке 20%.